# Feature Engineering & Data Preprocessing (Leakage-Free)

This notebook demonstrates how to clean, encode, and scale customer churn data. To prevent **data leakage**, we split our dataset into training and test sets *before* performing any preprocessing steps like imputation, scaling, or dummy encoding.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer

In [ ]:
df = pd.read_csv("/Users/anshrathore/Desktop/Customer Churn Prediction Project/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

### Step 1: Split into Train and Test Sets First
By splitting first, we ensure that statistical properties (like median for imputation, mean/std for scaling, and category sets for one-hot encoding) are learned solely from the training data.

In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

### Step 2: Clean the Data
- Drop customerID.
- Convert TotalCharges to numeric.
- Map Churn to binary values (0/1).

In [ ]:
# Drop customerID
X_train = X_train.drop("customerID", axis=1)
X_test = X_test.drop("customerID", axis=1)

# Convert TotalCharges to numeric (coercing spaces to NaN)
X_train["TotalCharges"] = pd.to_numeric(X_train["TotalCharges"], errors="coerce")
X_test["TotalCharges"] = pd.to_numeric(X_test["TotalCharges"], errors="coerce")

# Map target to binary
y_train = y_train.map({"Yes": 1, "No": 0})
y_test = y_test.map({"Yes": 1, "No": 0})

### Step 3: Fit Preprocessing Components on Train and Transform Train & Test

In [ ]:
# 1. Impute missing values in TotalCharges using the training set median
imputer = SimpleImputer(strategy='median')
X_train[["TotalCharges"]] = imputer.fit_transform(X_train[["TotalCharges"]])
X_test[["TotalCharges"]] = imputer.transform(X_test[["TotalCharges"]])

# 2. Scale continuous numerical columns using the training set mean and std
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# 3. Label encode binary categorical columns (unique values == 2)
bin_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
ordinal_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[bin_cols] = ordinal_enc.fit_transform(X_train[bin_cols])
X_test[bin_cols] = ordinal_enc.transform(X_test[bin_cols])

# 4. One-hot encode multi-category columns
multi_cols = [
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaymentMethod'
]
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
X_train_encoded = pd.DataFrame(
    ohe.fit_transform(X_train[multi_cols]), 
    columns=ohe.get_feature_names_out(multi_cols),
    index=X_train.index
)
X_test_encoded = pd.DataFrame(
    ohe.transform(X_test[multi_cols]),
    columns=ohe.get_feature_names_out(multi_cols),
    index=X_test.index
)

# Combine all processed features
X_train_final = pd.concat([X_train.drop(columns=multi_cols), X_train_encoded], axis=1)
X_test_final = pd.concat([X_test.drop(columns=multi_cols), X_test_encoded], axis=1)

print(f"Final X_train shape: {X_train_final.shape}")
print(f"Final X_test shape: {X_test_final.shape}")

### Step 4: Save Cleaned and Preprocessed Datasets separately

In [ ]:
# Save training set
train_df = X_train_final.copy()
train_df["Churn"] = y_train
train_df.to_csv("../data/processed/train_cleaned.csv", index=False)

# Save test set
test_df = X_test_final.copy()
test_df["Churn"] = y_test
test_df.to_csv("../data/processed/test_cleaned.csv", index=False)

print("Preprocessed datasets saved to data/processed/")